# T16 · Capa refinada con esquema declarado — cuaderno de control

Sesión 16 · Juan Pablo Castro, Camilo Rojas, Lina Ramírez

**Motor:** PySpark 3.5.3 nativo, Java 21, `local[2]`. **Equipo:** `equipo-2`.

In [ ]:
import os, sys, pathlib, subprocess
REPO = pathlib.Path('/Users/camilorojas/Projects/BigData/bigdata-ean-ideam')
PRACT = REPO / 'practica' / 's16-refinada-spark'
os.environ['JAVA_HOME'] = '/opt/homebrew/opt/openjdk@21'
os.environ['PYSPARK_PYTHON'] = str(REPO / '.venv-spark' / 'bin' / 'python')
PY = str(REPO / '.venv-spark' / 'bin' / 'python')
print('cwd =', os.getcwd())

## 0. Generar capas

In [ ]:
subprocess.run([PY, str(PRACT/'generar_capas_s16.py'), '--volumen', 'base'], cwd=str(REPO))
subprocess.run([PY, str(PRACT/'generar_capas_s16.py'), '--volumen', 'ampliacion', '--equipo', 'equipo-2'], cwd=str(REPO))

## 1. Capa refinada base (dos corridas)

In [ ]:
for _ in range(2):
    r = subprocess.run([PY, str(REPO/'pipeline'/'refinada_spark.py'), '--equipo','equipo-2','--volumen','base','--modo','refinada'],
                       cwd=str(REPO), capture_output=True, text=True)
    for line in r.stdout.splitlines():
        if line.startswith('RESULT'): print(line[:120])

## 2. Variantes de clasificación

In [ ]:
r = subprocess.run([PY, str(REPO/'pipeline'/'refinada_spark.py'), '--equipo','equipo-2','--volumen','ampliacion','--modo','variantes'],
                   cwd=str(REPO), capture_output=True, text=True)
for line in r.stdout.splitlines():
    if line.startswith('RESULT'): print(line[:200])

## 3. Reproceso y verificación

In [ ]:
for m in ['estatica','dinamica']:
    r = subprocess.run([PY, str(REPO/'pipeline'/'reproceso_spark.py'), '--equipo','equipo-2','--sobrescritura',m],
                       cwd=str(REPO), capture_output=True, text=True)
    for line in r.stdout.splitlines():
        if line.startswith('RESULT'): print(m, line[:150])

In [ ]:
subprocess.run([PY, str(PRACT/'generar_evidencia_t16.py'), '--equipo','equipo-2'], cwd=str(REPO))
r = subprocess.run([PY, str(REPO/'pipeline'/'verificar_etapa.py'), '--etapa','t16'], cwd=str(REPO), capture_output=True, text=True)
print(r.stdout)

## 4. Evidencia Nivel 2

In [ ]:
from IPython.display import Markdown, display
ev = (PRACT / 'resultados' / 'evidencia_nivel2_equipo-2.md').read_text()
display(Markdown(ev))